# Initial tests

In [2]:
# import libs
import jax
import jax.numpy as jnp

# define parameters
params = {'layer1': {'W': jnp.ones((3,4)), 'b': jnp.zeros(4)}, 'layer2': { 'W': jnp.ones((4,2)), 'b': jnp.zeros(2)}}

total = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total parameters: {total}")

scaled = jax.tree_util.tree_map(lambda x: x * 2, params)
print(f"Scaled layer1.W: {scaled['layer1']['W']}")

Total parameters: 26
Scaled layer1.W: [[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]


In [3]:
# Test the device
print('Devices:', jax.devices()); print('Backend:', jax.default_backend())

Devices: [CudaDevice(id=0)]
Backend: gpu


# JAX Basics for LLM Inference

This notebook covers the core JAX concepts we'll use to build nanospeed:
- JAX arrays (similar to NumPy, but with device awareness)
- `jit` for compilation
- `grad` for automatic differentiation
- `vmap` for automatic batching

These four primitives are the foundation of everything we build.

In [4]:
import jax
import jax.numpy as jnp
import numpy as np

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX version: 0.7.2
Devices: [CudaDevice(id=0)]
Default backend: gpu


In [5]:
# Functional style: create new arrays instead of modifying

x = jnp.array([1.0, 2.0, 3.0])
x_new = x.at[0].set(99.0)  # Returns a NEW array
print("Original:", x)
print("Modified:", x_new)

# This functional style is crucial for jax.jit and jax.grad
# It enables JAX's transformations to work correctly

Original: [1. 2. 3.]
Modified: [99.  2.  3.]


In [6]:
# JAX arrays are similar to NumPy but live on devices (GPU/TPU/CPU)

# NumPy array
x_np = np.array([1.0, 2.0, 3.0])
print("NumPy array:", x_np, "type:", type(x_np))

# JAX array
x_jax = jnp.array([1.0, 2.0, 3.0])
print("JAX array:", x_jax, "type:", type(x_jax))

# They feel similar, but JAX arrays are immutable!
# This will raise an error:
try:
    x_jax[0] = 99.0
except TypeError as e:
    print("Error:", e)

NumPy array: [1. 2. 3.] type: <class 'numpy.ndarray'>
JAX array: [1. 2. 3.] type: <class 'jaxlib._jax.ArrayImpl'>
Error: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html


# JIT Compilation

`jax.jit` transforms a Python function into a compiled, optimized version.

Why this matters for LLM inference:
- The forward pass of a transformer is mostly matrix multiplications
- Compiled code runs 10-100× faster than Python interpretation
- For our KV cache, we'll jit-compile the attention step itself

The trade-off:
- First call is slow (compilation overhead)
- Subsequent calls are fast (cached compilation)
- JAX needs to "see" the function's structure before it can optimize

In [7]:
# A simple function without jit
def slow_multiply(x, y):
    return jnp.sin(x) * jnp.cos(y)

# Same function with jit
fast_multiply = jax.jit(slow_multiply)

# Create some data
x = jnp.array([1.0, 2.0, 3.0])
y = jnp.array([0.5, 1.0, 1.5])

# Time both
import time

# First call with jit includes compilation
start = time.time()
result_jit_1 = fast_multiply(x, y)
result_jit_1.block_until_ready()  # Wait for GPU/CPU to finish
time_jit_1 = time.time() - start
print(f"First JIT call (includes compilation): {time_jit_1*1000:.2f} ms")

# Second call with jit (cached)
start = time.time()
result_jit_2 = fast_multiply(x, y)
result_jit_2.block_until_ready()
time_jit_2 = time.time() - start
print(f"Second JIT call (cached): {time_jit_2*1000:.4f} ms")

# Without jit
start = time.time()
result_no_jit = slow_multiply(x, y)
result_no_jit.block_until_ready()
time_no_jit = time.time() - start
print(f"Without JIT: {time_no_jit*1000:.4f} ms")

print(f"\nSpeedup from JIT: {time_no_jit/time_jit_2:.1f}x")

First JIT call (includes compilation): 89.16 ms
Second JIT call (cached): 0.5996 ms
Without JIT: 142.8685 ms

Speedup from JIT: 238.3x


In [8]:
# Important: JIT specializes on shape and dtype

# This works (same shape and dtype)
fast_multiply(jnp.array([2.0, 3.0, 4.0]), jnp.array([1.0, 0.5, 0.2]))

# This triggers RECOMPILATION (different shape)
try:
    start = time.time()
    fast_multiply(jnp.array([2.0, 3.0]), jnp.array([1.0, 0.5])).block_until_ready()
    print("Time for this is ", time.time()-start)
except Exception as e:
    print("Different shape triggered recompilation")

# This triggers RECOMPILATION (different dtype)
try:
    start = time.time()
    fast_multiply(jnp.array([2, 3, 4]), jnp.array([1, 2, 3])).block_until_ready()   # int instead of float
    print("Time required for this is ", time.time()-start)
except Exception as e:
    print("Different dtype triggered recompilation")

Time for this is  0.09050846099853516
Time required for this is  0.08883142471313477


# Automatic Batching with vmap

`jax.vmap` automatically batches a function that works on a single example
into one that works on many examples.

This is CRUCIAL for LLM inference serving:
- We don't process one user request at a time — we batch many together
- vmap lets us write code for a single token and automatically parallelize
- Without vmap, we'd need explicit loops or complex indexing

In [9]:
# Function that works on a single vector
def square(x):
    return x ** 2

# Apply to single vector
x_single = jnp.array([1.0, 2.0, 3.0])
print("Single:", square(x_single))

# Apply to batch of vectors using vmap
x_batch = jnp.array([[1.0, 2.0, 3.0],
                     [4.0, 5.0, 6.0],
                     [7.0, 8.0, 9.0]])  # shape (3, 3)

# vmap automatically adds a batch dimension
batched_square = jax.vmap(square)
result = batched_square(x_batch)
print("Batched shape:", result.shape)
print("Batched result:", result)

Single: [1. 4. 9.]
Batched shape: (3, 3)
Batched result: [[ 1.  4.  9.]
 [16. 25. 36.]
 [49. 64. 81.]]


In [10]:
# More realistic example: matrix-vector product

def matvec(W, x):
    """Single example: W is (d_out, d_in), x is (d_in,)"""
    return W @ x

# Batch version: W stays (d_out, d_in), x becomes (batch, d_in)
W = jnp.array([[1.0, 2.0],
               [3.0, 4.0],
               [5.0, 6.0]])  # (3, 2)

x_batch = jnp.array([[1.0, 1.0],
                     [2.0, 2.0],
                     [3.0, 3.0]])  # (3, 2) — batch of 3

batched_matvec = jax.vmap(matvec)
result = batched_matvec(W, x_batch)
print("Result shape:", result.shape)
print("Result:\n", result)

Result shape: (3,)
Result:
 [ 3. 14. 33.]


In [11]:
# Combining jit + vmap for maximum performance

@jax.jit
@jax.vmap
def fast_square(x):
    return x ** 2

x_batch = jnp.arange(12).reshape(4, 3).astype(jnp.float32)

# First call: compilation
import time
start = time.time()
result = fast_square(x_batch).block_until_ready()
print(f"First call (compile): {(time.time()-start)*1000:.2f} ms")

# Second call: fast
start = time.time()
result = fast_square(x_batch).block_until_ready()
print(f"Second call: {(time.time()-start)*1000:.4f} ms")

First call (compile): 38.44 ms
Second call: 0.4990 ms


# Exercises

## Exercise 1: JAX Arrays + Immutability

**Task:** Given `x = jnp.array([10.0, 20.0, 30.0, 40.0])`:
- Try to modify `x` in-place (this should fail). Show the error.
- Create a new array `y` where element at index 2 is changed to 99.0
- Verify that `x` is unchanged after creating `y`
- Compute the sum of `x` and `y` and store it in `z`

In [12]:
x = jnp.array([10.0, 20.0, 30.0, 40.0])
print(x)

[10. 20. 30. 40.]


In [13]:
#x[0] = 50
y = x.at[0].set(50)
print(y)
print(x)

[50. 20. 30. 40.]
[10. 20. 30. 40.]


In [14]:
import jax
import jax.numpy as jnp
import time

# Make 1000 vectors of size 128
x = jnp.ones((1000, 128))
y = jnp.ones((1000, 128))

# Method 1: Naive loop
def add(x, y):
    return x + y

start = time.time()
results = []
for i in range(1000):
    results.append(add(x[i], y[i]))
results = jnp.stack(results)
results.block_until_ready()
naive_time = time.time() - start
print(f"Naive loop: {naive_time*1000:.2f} ms")

# Method 2: vmap
batched_add = jax.jit(jax.vmap(add))

start = time.time()
result = batched_add(x, y)
result.block_until_ready()
vmap_time = time.time() - start
print(f"Vmap+JIT:   {vmap_time*1000:.2f} ms")

print(f"Speedup: {naive_time/vmap_time:.0f}x")

Naive loop: 1727.54 ms
Vmap+JIT:   45.24 ms
Speedup: 38x


In [15]:
def add(x,y):
    return x+y

z = jax.vmap(add)(x,y)
print(z)

[[2. 2. 2. ... 2. 2. 2.]
 [2. 2. 2. ... 2. 2. 2.]
 [2. 2. 2. ... 2. 2. 2.]
 ...
 [2. 2. 2. ... 2. 2. 2.]
 [2. 2. 2. ... 2. 2. 2.]
 [2. 2. 2. ... 2. 2. 2.]]


## Exercise 2: JIT Compilation + Timing

**Task:** Write two versions of a function that computes the sum of elements in a vector:
- `sum_naive(x)`: uses a Python `for` loop to sum elements
- `sum_jit(x)`: uses `jax.jit` and `jnp.sum`

Compare their performance on a vector of size 1,000,000:
- Time `sum_naive` 100 times, report average
- Time `sum_jit` 100 times (after one warmup call), report average
- Report the speedup factor

In [16]:
# For a vector of 100, since 1000000 is taking too much time
def sum_naive(x):
    ans = 0;
    for i in x:
        ans+=i
    return ans

@jax.jit
def sum_jit(x):
    return jnp.sum(x)

vec = jnp.ones(100)

start = time.time()
for i in range(100):
    sum_naive(vec).block_until_ready()
print("Time needed to run sum_naive 100 times: ",time.time()-start)

#sum_jit initial run
sum_jit(vec)

start = time.time()
for i in range(100):
    sum_jit(vec).block_until_ready()
print("Time needed to run sum_jit 100 times: ",time.time()-start)

Time needed to run sum_naive 100 times:  1.6255033016204834
Time needed to run sum_jit 100 times:  0.013564109802246094


In [17]:
# For a vector of 1000
vec = jnp.ones(1000)

start = time.time()
for i in range(100):
    sum_naive(vec).block_until_ready()
print("Time needed to run sum_naive 100 times: ",time.time()-start)

#sum_jit initial run
sum_jit(vec)

start = time.time()
for i in range(100):
    sum_jit(vec).block_until_ready()
print("Time needed to run sum_jit 100 times: ",time.time()-start)

Time needed to run sum_naive 100 times:  9.26383662223816
Time needed to run sum_jit 100 times:  0.012519359588623047


In [18]:
# For a vector of 10000
vec = jnp.ones(10000)

start = time.time()
for i in range(100):
    sum_naive(vec).block_until_ready()
print("Time needed to run sum_naive 100 times: ",time.time()-start)

#sum_jit initial run
sum_jit(vec)

start = time.time()
for i in range(100):
    sum_jit(vec).block_until_ready()
print("Time needed to run sum_jit 100 times: ",time.time()-start)

Time needed to run sum_naive 100 times:  90.52307343482971
Time needed to run sum_jit 100 times:  0.012084722518920898


## Exercise 3: Vmap Batching

**Task:** Write a function `relu(x)` that returns `max(0, x)` element-wise.

Then:
1. Apply it to a single vector of shape `(5,)`
2. Use `jax.vmap` to apply it to a batch of 10 vectors, each of shape `(5,)`
3. Verify the output shape is `(10, 5)`
4. JIT-compile the vmap'd version and verify it gives the same result

**Hint:** Use `jnp.maximum(x, 0)` for the ReLU operation.

In [24]:
@jax.vmap
def relu(x):
    return jnp.maximum(0,x)

x = jnp.array(np.random.randn(5)*10)
print(x)

[  3.4645433   -0.820276    -0.07618019  -3.1174157  -11.730487  ]


In [25]:
relu(x)

Array([3.4645433, 0.       , 0.       , 0.       , 0.       ], dtype=float32)

In [29]:
# Creating a batch of 10
x_batch = jnp.array([jnp.array(np.random.randn(5)*10) for i in range(10)])
print(x_batch)

[[  1.7366588   18.640753    -6.2166705   17.890343    11.9815035 ]
 [ -0.05572849 -25.30285      3.8475764   -3.1074212   -8.597694  ]
 [-10.910417    -8.595674   -13.3506365   -0.5379461   -1.6779062 ]
 [  6.3242993    9.800432    -3.0067344    1.3274745   10.516994  ]
 [ 10.335252     4.02319    -16.707388   -13.735201    -6.765379  ]
 [ -4.3212943   -7.715981     1.2384114    0.27156007   1.7640933 ]
 [ 20.231564    -0.04280229   5.2014136   -5.7950144   -8.445381  ]
 [ -9.864003     5.891372     7.6134596   -9.538465    -0.92242897]
 [-13.706358     0.40599465  -3.4680667   -4.9619484   -8.466536  ]
 [ -4.3397202   11.743272    -6.982738   -15.680327    -8.302645  ]]


In [37]:
relu_batch = jax.vmap(relu)(x_batch)
print(relu_batch)

[[ 1.7366588  18.640753    0.         17.890343   11.9815035 ]
 [ 0.          0.          3.8475764   0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 6.3242993   9.800432    0.          1.3274745  10.516994  ]
 [10.335252    4.02319     0.          0.          0.        ]
 [ 0.          0.          1.2384114   0.27156007  1.7640933 ]
 [20.231564    0.          5.2014136   0.          0.        ]
 [ 0.          5.891372    7.6134596   0.          0.        ]
 [ 0.          0.40599465  0.          0.          0.        ]
 [ 0.         11.743272    0.          0.          0.        ]]


## Exercise 4: JIT + Vmap Combined

**Task:** Write a function that computes the dot product of two vectors. Use `jax.vmap` to batch over many pairs of vectors. JIT-compile the whole pipeline. Time it against a naive Python loop implementation.

- Generate 1000 random vectors of dimension 128
- Compute 1000 dot products two ways:
  - Naive: Python loop calling `jnp.dot` 1000 times
  - Optimized: `jax.vmap(jax.jit(jnp.dot))` called once
- Report the speedup

In [38]:
def dot_product(vec1,vec2):
    return vec1 @ vec2

vec1 = jnp.array([8.0, 7.0, 5.0, 3.0, 1.0])
vec2 = jnp.array([1.0, 3.0, 5.0, 7.0, 9.0])

print(dot_product(vec1, vec2))

84.0


## Exercise 5: Functional Style (Bonus)

**Task:** Given a JAX array `x` of shape `(10,)`:
- Replace all negative values with 0 (using `.at[].set()` or boolean masking)
- Verify the original `x` is unchanged
- Show that the new array has the same shape and dtype as `x`

In [21]:
print(vec.device)

cuda:0


In [22]:
import jax
import jax.numpy as jnp
import numpy as np
import time

def slow_function(x):
    return jnp.sin(x) @ jnp.cos(x.T)

# NumPy version
x_np = np.random.randn(1000, 1000)
start = time.time()
for _ in range(10):
    _ = np.sin(x_np) @ np.cos(x_np.T)
numpy_time = time.time() - start

# JAX version (no JIT)
x_jax = jnp.array(x_np)
start = time.time()
for _ in range(10):
    _ = slow_function(x_jax).block_until_ready()
jax_no_jit = time.time() - start

# JAX version (with JIT)
fast = jax.jit(slow_function)
_ = fast(x_jax).block_until_ready()  # compile once
start = time.time()
for _ in range(10):
    _ = fast(x_jax).block_until_ready()
jax_jit = time.time() - start

print(f"NumPy:    {numpy_time:.3f}s")
print(f"JAX:      {jax_no_jit:.3f}s")
print(f"JAX+JIT:  {jax_jit:.3f}s")

NumPy:    0.690s
JAX:      0.379s
JAX+JIT:  0.006s
